# Team 11 · Notebook 03: Machine Learning Models and Evaluation
**Project:** Road Accident Risk Pipeline for usage-based auto insurance  
**Inputs:** `silver_accidents` and `gold_monthly_state_volume`  
**Tracking:** every model run is logged in **MLflow** (parameters, metrics and charts), and all metrics are saved to the `gold_model_metrics` table for the dashboard.

| | Model A: Severity risk (classification) | Model B: Accident volume forecast (regression) |
|---|---|---|
| **Business question** | Given the time, place, weather and road, how likely is an accident to be **severe** (level 3–4)? | How many accidents will each **state** have **next month**? |
| **Who uses it** | Pricing team: risk factors for premiums and safety advice | Claims team: plan staff and reserves by state |
| **Algorithms (Spark MLlib)** | Logistic regression, Random forest | Linear regression, Random forest |
| **Compared against** | Always guessing "not severe" | Next month = this month (naive forecast) |
| **Metrics** | ROC-AUC, PR-AUC, accuracy, precision, recall, F1, confusion matrix | R², RMSE, MAE, MAPE |

**Gaps this closes:** Spark/PySpark on large data (MLlib), lakehouse layers (gold feeds the models), data quality (leakage and imbalance checks), monitoring workflows (MLflow tracking).

## 0. Setup
`ML_SAMPLE_FRACTION = 0.10` trains Model A on a fixed 10% sample (about 770,000 accidents). That is plenty for a stable model and keeps us inside the daily compute quota. As we learned in Checkpoint 3, the sample is saved once as a table so every step uses exactly the same rows.

In [ ]:
from pyspark.sql import functions as F, Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import BinaryClassificationEvaluator, RegressionEvaluator
import numpy as np, pandas as pd, os, time
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from sklearn.metrics import (roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
                             confusion_matrix, accuracy_score, precision_score, recall_score, f1_score,
                             r2_score, mean_squared_error, mean_absolute_error)

CATALOG = "workspace"
SCHEMA  = "accidents"
FIG_DIR = "/Volumes/workspace/accidents/figures"
ML_SAMPLE_FRACTION = 0.10
SEED = 42

def tbl(name):
    return f"{CATALOG}.{SCHEMA}.{name}"
os.makedirs(FIG_DIR, exist_ok=True)

try:
    import mlflow
    mlflow.autolog(disable=True)   # we log exactly what we want, by hand
    MLFLOW = True
except Exception as e:
    MLFLOW = False
    print("MLflow not available here:", str(e)[:100])

BLUE, AQUA, ORANGE, GREY, INK, MUTED = "#2a78d6", "#1baf7a", "#eb6834", "#9a9893", "#1f1f1e", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.edgecolor": "#c9c7c1", "axes.labelcolor": MUTED, "xtick.color": MUTED,
    "ytick.color": MUTED, "text.color": INK, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#ebe9e4", "grid.linewidth": 0.8, "axes.axisbelow": True,
})
def save(fig, name):
    fig.tight_layout()
    path = f"{FIG_DIR}/{name}.png"
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    return path

all_metrics = []   # (model, task, split, metric, value) -> saved to gold_model_metrics at the end
def record(model, task, split, metrics):
    for k, v in metrics.items():
        all_metrics.append((model, task, split, k, float(v)))

def log_run(name, params, metrics, figures):
    # Log one model run to MLflow; the notebook keeps working even if MLflow is unavailable
    if not MLFLOW:
        return
    try:
        with mlflow.start_run(run_name=name):
            mlflow.log_params(params)
            mlflow.log_metrics({k: float(v) for k, v in metrics.items()})
            for f in figures:
                mlflow.log_artifact(f)
        print(f"MLflow: logged run '{name}'")
    except Exception as e:
        print("MLflow logging skipped:", str(e)[:150])

# Part A: Severity risk model (classification)

## A1. Build the modelling table
**Target:** `label` = 1 if the accident is severe (level 3–4), else 0.

**Inputs we use:** hour, weekday, month, day or night, state, weather category, temperature, humidity, pressure, visibility, wind speed, precipitation, and the 13 road features (junction, traffic signal, crossing and so on).

**Inputs we deliberately leave out, to avoid leakage** (letting the model see the answer):
- `Distance_mi` and `duration_min`: how long the road was blocked is a *result* of how severe the accident was.
- `Description`: the free text often describes the outcome.
- `year` and `Source`: they reflect how the data was collected over time, not real risk.

Missing weather values are filled with the median inside the model, and we add a flag for missing precipitation (missing in about 28% of rows) so the model can learn from the gap itself.

In [ ]:
road_features = ["Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit", "Railway",
                 "Roundabout", "Station", "Stop", "Traffic_Calming", "Traffic_Signal", "Turning_Loop"]
num_cols = ["Temperature_F", "Humidity_pct", "Pressure_in", "Visibility_mi", "Wind_Speed_mph", "Precipitation_in"]

(spark.table(tbl("silver_accidents"))
    .sample(fraction=ML_SAMPLE_FRACTION, seed=SEED)
    .select(
        F.col("is_severe").cast("int").alias("label"),
        "hour", "weekday_num", "month",
        F.when(F.col("Sunrise_Sunset") == "Night", 1).otherwise(0).alias("is_night"),
        F.coalesce(F.col("State"), F.lit("UNKNOWN")).alias("State"),
        "weather_category",
        *[F.col(c).cast("double").alias(c) for c in num_cols],
        F.col("Precipitation_in").isNull().cast("int").alias("precip_missing"),
        *[F.coalesce(F.col(c), F.lit(False)).cast("int").alias(c) for c in road_features])
    .filter(F.col("label").isNotNull())
    .withColumn("split_rand", F.rand(SEED))              # fixed train/test assignment, saved with the data
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("gold_ml_severity_sample")))

ml = spark.table(tbl("gold_ml_severity_sample"))
train = ml.filter(F.col("split_rand") < 0.8)
test  = ml.filter(F.col("split_rand") >= 0.8)

n_train, n_test = train.count(), test.count()
pos_rate = train.agg(F.avg("label")).first()[0]
print(f"Training rows: {n_train:,} | Test rows: {n_test:,}")
print(f"Severe share in training data: {100*pos_rate:.2f}%  ->  the classes are imbalanced")

## A2. Handle the imbalance and build the two models
Only a minority of accidents are severe. A model could score high **accuracy** just by always saying "not severe", while catching none of the severe ones. So we:
1. **Weight the classes** so severe and non-severe accidents count equally during training.
2. Judge the models mainly on **ROC-AUC, PR-AUC, recall and F1**, not accuracy alone.

Both models share the same preparation steps (a Spark ML **Pipeline**): turn state and weather into numbers (indexing and one-hot encoding), fill missing values with the median, and combine all inputs into one feature vector.

In [ ]:
w_pos, w_neg = 0.5 / pos_rate, 0.5 / (1 - pos_rate)
add_weight = lambda df: df.withColumn("weight", F.when(F.col("label") == 1, w_pos).otherwise(w_neg))
train_w = add_weight(train)

cat_cols = ["State", "weather_category"]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
encoder  = OneHotEncoder(inputCols=[f"{c}_idx" for c in cat_cols], outputCols=[f"{c}_ohe" for c in cat_cols], dropLast=False)
imputer  = Imputer(inputCols=num_cols, outputCols=[f"{c}_imp" for c in num_cols], strategy="median")
base_cols = ["hour", "weekday_num", "month", "is_night", "precip_missing"] + [f"{c}_imp" for c in num_cols] + road_features
assembler = VectorAssembler(inputCols=base_cols + [f"{c}_ohe" for c in cat_cols], outputCol="features")
prep = indexers + [encoder, imputer, assembler]

lr = LogisticRegression(featuresCol="features_scaled", labelCol="label", weightCol="weight",
                        maxIter=50, regParam=0.01)
rf = RandomForestClassifier(featuresCol="features", labelCol="label", weightCol="weight",
                            numTrees=50, maxDepth=10, subsamplingRate=0.8, seed=SEED)

t = time.time()
lr_model = Pipeline(stages=prep + [StandardScaler(inputCol="features", outputCol="features_scaled"), lr]).fit(train_w)
lr_secs = round(time.time() - t, 1)
t = time.time()
rf_model = Pipeline(stages=prep + [rf]).fit(train_w)
rf_secs = round(time.time() - t, 1)
print(f"Training time: logistic regression {lr_secs}s | random forest {rf_secs}s")

## A3. Evaluate on the test set (data the models never saw)
**How to read the metrics**
- **ROC-AUC:** the chance the model ranks a random severe accident above a random non-severe one. 0.5 = guessing, 1.0 = perfect.
- **PR-AUC:** how well the model finds severe accidents; the "guessing" level equals the severe share.
- **Precision:** of the accidents flagged as severe, how many really were.
- **Recall:** of all severe accidents, how many the model caught.
- **F1:** the balance of precision and recall.

In [ ]:
try:
    from pyspark.ml.functions import vector_to_array
    p_severe = lambda: vector_to_array("probability")[1]
except Exception:
    p_severe = lambda: F.udf(lambda v: float(v[1]), "double")("probability")

def scores(model, df):
    return model.transform(df).select("label", p_severe().alias("p")).toPandas()

pred = {"Logistic regression": scores(lr_model, test), "Random forest": scores(rf_model, test)}
y = pred["Random forest"]["label"].values

def class_metrics(y_true, p, threshold=0.5):
    yhat = (p >= threshold).astype(int)
    return {"roc_auc": roc_auc_score(y_true, p), "pr_auc": average_precision_score(y_true, p),
            "accuracy": accuracy_score(y_true, yhat), "precision": precision_score(y_true, yhat, zero_division=0),
            "recall": recall_score(y_true, yhat), "f1": f1_score(y_true, yhat)}

baseline = {"roc_auc": 0.5, "pr_auc": float(y.mean()), "accuracy": float(1 - y.mean()),
            "precision": 0.0, "recall": 0.0, "f1": 0.0}
results = {"Baseline: always 'not severe'": baseline}
for name, d in pred.items():
    results[name] = class_metrics(d["label"].values, d["p"].values)

# Training-set AUC from Spark's own evaluator, to check for overfitting
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
train_auc = {"Logistic regression": evaluator.evaluate(lr_model.transform(train)),
             "Random forest": evaluator.evaluate(rf_model.transform(train))}

table = pd.DataFrame(results).T.round(3)
table["train_roc_auc"] = [None, round(train_auc["Logistic regression"], 3), round(train_auc["Random forest"], 3)]
display(spark.createDataFrame(table.reset_index().rename(columns={"index": "model"})))

In [ ]:
# Chart A1: ROC curves
fig, ax = plt.subplots(figsize=(5.6, 4.6))
for (name, d), c in zip(pred.items(), [BLUE, AQUA]):
    fpr, tpr, _ = roc_curve(d["label"], d["p"])
    ax.plot(fpr, tpr, color=c, lw=2, label=f"{name} (AUC = {results[name]['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], color=GREY, ls="--", lw=1.2, label="Random guess (AUC = 0.500)")
ax.set_xlabel("False positive rate (non-severe flagged as severe)")
ax.set_ylabel("True positive rate (recall)")
ax.set_title("ROC curve: severity risk model (test set)")
ax.legend(frameon=False, loc="lower right", fontsize=8)
f_roc = save(fig, "A1_roc_curve"); display(fig)

In [ ]:
# Chart A2: precision-recall curves
fig, ax = plt.subplots(figsize=(5.6, 4.6))
for (name, d), c in zip(pred.items(), [BLUE, AQUA]):
    prec, rec, _ = precision_recall_curve(d["label"], d["p"])
    ax.plot(rec, prec, color=c, lw=2, label=f"{name} (PR-AUC = {results[name]['pr_auc']:.3f})")
ax.axhline(y.mean(), color=GREY, ls="--", lw=1.2, label=f"Random guess ({y.mean():.3f})")
ax.set_xlabel("Recall (severe accidents caught)"); ax.set_ylabel("Precision (flags that were right)")
ax.set_title("Precision-recall curve (test set)")
ax.set_ylim(0, 1.02); ax.legend(frameon=False, fontsize=8)
f_pr = save(fig, "A2_precision_recall_curve"); display(fig)

In [ ]:
# Chart A3: confusion matrices at the 0.5 threshold
fig, axes = plt.subplots(1, 2, figsize=(9, 3.9))
for ax, (name, d) in zip(axes, pred.items()):
    cm = confusion_matrix(d["label"], (d["p"] >= 0.5).astype(int))
    pct = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(pct, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}\n({pct[i, j]:.0%} of row)", ha="center", va="center",
                    color="white" if pct[i, j] > 0.55 else INK, fontsize=9)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Predicted\nnot severe", "Predicted\nsevere"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Actually\nnot severe", "Actually\nsevere"])
    ax.set_title(name, fontsize=11)
fig.suptitle("Confusion matrices (test set, threshold 0.5)", x=0.01, ha="left", fontweight="bold")
f_cm = save(fig, "A3_confusion_matrices"); display(fig)

In [ ]:
# Chart A4: what drives severity? Random forest feature importance (top 15)
names = list(base_cols)
try:
    for i, c in enumerate(cat_cols):
        labels = list(rf_model.stages[i].labels)
        names += [f"{c.replace('weather_category', 'Weather')}={l}" for l in labels] + [f"{c}=other"]
except Exception as e:
    print("Category names not available:", str(e)[:100])
imp = rf_model.stages[-1].featureImportances.toArray()
if len(names) != len(imp):
    names = [f"feature_{i}" for i in range(len(imp))]
fi = (pd.DataFrame({"feature": [n.replace("_imp", "") for n in names], "importance": imp})
        .sort_values("importance", ascending=False).head(15)[::-1])
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(fi["feature"], fi["importance"], color=AQUA, height=0.6)
ax.set_title("What drives severity: top 15 inputs (random forest)")
ax.set_xlabel("Share of the model's decisions (feature importance)"); ax.grid(axis="y", visible=False)
f_fi = save(fig, "A4_feature_importance"); display(fig)
display(spark.createDataFrame(fi[::-1].round(4)))

In [ ]:
# Chart A5: choosing a threshold - the business trade-off between catching severe accidents and false alarms
d = pred["Random forest"]
th = np.linspace(0.1, 0.9, 33)
P = [precision_score(d["label"], (d["p"] >= t).astype(int), zero_division=0) for t in th]
R = [recall_score(d["label"], (d["p"] >= t).astype(int)) for t in th]
F1 = [f1_score(d["label"], (d["p"] >= t).astype(int)) for t in th]
best_t = th[int(np.argmax(F1))]
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(th, P, color=BLUE, lw=2, label="Precision")
ax.plot(th, R, color=ORANGE, lw=2, label="Recall")
ax.plot(th, F1, color=AQUA, lw=2, label="F1")
ax.axvline(best_t, color=GREY, ls="--", lw=1.2)
ax.text(best_t, 0.03, f" best F1 at {best_t:.2f}", color=MUTED, fontsize=8)
ax.set_xlabel("Probability threshold for flagging 'severe'"); ax.set_ylabel("Score")
ax.set_title("Random forest: precision vs recall by threshold"); ax.set_ylim(0, 1.02)
ax.legend(frameon=False, fontsize=8)
f_th = save(fig, "A5_threshold_tradeoff"); display(fig)
print(f"Threshold with the best F1: {best_t:.2f} -> F1 {max(F1):.3f}, precision {P[int(np.argmax(F1))]:.3f}, recall {R[int(np.argmax(F1))]:.3f}")

In [ ]:
# Record and track Part A
for name in ["Logistic regression", "Random forest"]:
    m = dict(results[name]); m["train_roc_auc"] = train_auc[name]
    record(name, "severity_classification", "test", m)
record("Baseline: always 'not severe'", "severity_classification", "test", baseline)

common = {"sample_fraction": ML_SAMPLE_FRACTION, "train_rows": n_train, "test_rows": n_test,
          "severe_share_train": round(pos_rate, 4), "class_weighting": "balanced"}
figs = [f_roc, f_pr, f_cm]
log_run("A_logistic_regression", {**common, "model": "LogisticRegression", "maxIter": 50, "regParam": 0.01, "train_seconds": lr_secs},
        {**results["Logistic regression"], "train_roc_auc": train_auc["Logistic regression"]}, figs)
log_run("A_random_forest", {**common, "model": "RandomForestClassifier", "numTrees": 50, "maxDepth": 10, "train_seconds": rf_secs},
        {**results["Random forest"], "train_roc_auc": train_auc["Random forest"], "best_f1_threshold": float(best_t)}, figs + [f_fi, f_th])

# Part B: Accident volume forecast (regression)

## B1. Build the monthly table
We start from `gold_monthly_state_volume` and fill in a **complete grid** of every state and every month (a month with no accidents becomes 0, instead of silently disappearing). For each state and month, the model sees:
- accidents in each of the previous 3 months (`lag1`, `lag2`, `lag3`)
- accidents in the same month last year (`lag12`), and how much they changed from the month before last year (seasonality)
- the 3-month average, last month's change, the calendar month, and the state

**What the models predict:** the **change** from last month, which is then added to last month's value. Tree-based models cannot predict numbers larger than any they saw in training, and accident volumes grow over the years, so predicting the change (rather than the raw count) lets them follow that growth.

**Time-based split:** the model trains on older months and is tested on the **last 12 months** of data, exactly as it would be used in real life (forecasting the future from the past). A random split would let it peek at the future.

In [ ]:
mv = spark.table(tbl("gold_monthly_state_volume")).select("State", "year_month", "accidents")
months = mv.select("year_month").distinct()
states = mv.select("State").distinct()
grid = (states.crossJoin(months).join(mv, ["State", "year_month"], "left")
        .withColumn("accidents", F.coalesce("accidents", F.lit(0)).cast("double")))

w = Window.partitionBy("State").orderBy("year_month")
monthly = (grid
    .withColumn("lag1", F.lag("accidents", 1).over(w))
    .withColumn("lag2", F.lag("accidents", 2).over(w))
    .withColumn("lag3", F.lag("accidents", 3).over(w))
    .withColumn("lag12", F.lag("accidents", 12).over(w))
    .withColumn("lag13", F.lag("accidents", 13).over(w))
    .withColumn("roll3", (F.col("lag1") + F.col("lag2") + F.col("lag3")) / 3)
    .withColumn("change_last_month", F.col("lag1") - F.col("lag2"))
    .withColumn("change_same_month_last_year", F.col("lag12") - F.col("lag13"))
    .withColumn("month", F.month("year_month"))
    .withColumn("change", F.col("accidents") - F.col("lag1"))      # what the models predict
    .dropna(subset=["lag1", "lag2", "lag3", "lag12", "lag13"]))

last_month = monthly.agg(F.max("year_month")).first()[0]
cutoff = monthly.select(F.add_months(F.lit(last_month), -12)).first()[0]
mtrain = monthly.filter(F.col("year_month") <= F.lit(cutoff))
mtest  = monthly.filter(F.col("year_month") > F.lit(cutoff))
print(f"Train: up to {cutoff} ({mtrain.count():,} state-months) | Test: after {cutoff} to {last_month} ({mtest.count():,} state-months)")

## B2. Train and evaluate
**How to read the metrics**
- **R²:** how much of the month-to-month variation the model explains. 1.0 = perfect, 0 = no better than predicting the average.
- **RMSE / MAE:** typical size of the error, in accidents per state per month (RMSE punishes big misses more).
- **MAPE:** average error as a percentage of the actual value (months with zero accidents are skipped).

We compare against a **naive forecast**: "next month will be the same as this month". A model is only useful if it beats that.

In [ ]:
reg_inputs = ["lag1", "lag2", "lag3", "lag12", "roll3", "change_last_month", "change_same_month_last_year", "month"]
s_idx = StringIndexer(inputCol="State", outputCol="State_idx", handleInvalid="keep")
s_ohe = OneHotEncoder(inputCols=["State_idx"], outputCols=["State_ohe"], dropLast=False)
r_asm = VectorAssembler(inputCols=reg_inputs + ["State_ohe"], outputCol="features")

linreg = Pipeline(stages=[s_idx, s_ohe, r_asm,
                          LinearRegression(featuresCol="features", labelCol="change", regParam=0.1, elasticNetParam=0.0)]).fit(mtrain)
rfreg = Pipeline(stages=[s_idx, s_ohe, r_asm,
                         RandomForestRegressor(featuresCol="features", labelCol="change", numTrees=100, maxDepth=6, seed=SEED)]).fit(mtrain)

def reg_metrics(y_true, y_pred):
    nz = y_true > 0
    return {"r2": r2_score(y_true, y_pred), "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "mae": mean_absolute_error(y_true, y_pred),
            "mape_pct": float(np.mean(np.abs((y_true[nz] - y_pred[nz]) / y_true[nz])) * 100) if nz.any() else float("nan")}

def reg_scores(model, df):
    # forecast = last month's accidents + predicted change (never below zero)
    return (model.transform(df)
            .withColumn("prediction", F.greatest(F.lit(0.0), F.col("lag1") + F.col("prediction")))
            .select("State", "year_month", "accidents", "lag1", "prediction").toPandas())

rpred = {"Linear regression": reg_scores(linreg, mtest), "Random forest": reg_scores(rfreg, mtest)}
rtrain = {"Linear regression": reg_scores(linreg, mtrain), "Random forest": reg_scores(rfreg, mtrain)}
naive = rpred["Random forest"]
reg_results = {"Naive: next month = this month": reg_metrics(naive["accidents"].values, naive["lag1"].values)}
for name in rpred:
    reg_results[name] = reg_metrics(rpred[name]["accidents"].values, rpred[name]["prediction"].values)
    reg_results[name]["train_r2"] = r2_score(rtrain[name]["accidents"].values, rtrain[name]["prediction"].values)

rtable = pd.DataFrame(reg_results).T.round(3)
display(spark.createDataFrame(rtable.reset_index().rename(columns={"index": "model"})))

In [ ]:
# Chart B1: actual vs predicted (test months)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2), sharex=True, sharey=True)
for ax, (name, d), c in zip(axes, rpred.items(), [BLUE, AQUA]):
    ax.scatter(d["accidents"], d["prediction"], s=14, color=c, alpha=0.6, edgecolors="none")
    lim = max(d["accidents"].max(), d["prediction"].max()) * 1.05
    ax.plot([0, lim], [0, lim], color=GREY, ls="--", lw=1.2)
    ax.set_title(f"{name}: R² = {reg_results[name]['r2']:.3f}", fontsize=11)
    ax.set_xlabel("Actual accidents (state-month)")
axes[0].set_ylabel("Predicted accidents")
fig.suptitle("Accident volume forecast: actual vs predicted (test months)", x=0.01, ha="left", fontweight="bold")
f_avp = save(fig, "B1_actual_vs_predicted"); display(fig)

In [ ]:
# Chart B2: forecast vs actual over time for the 4 busiest states
top4 = rpred["Random forest"].groupby("State")["accidents"].sum().nlargest(4).index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(10, 5.6), sharex=True)
for ax, st in zip(axes.flat, top4):
    d = rpred["Random forest"][rpred["Random forest"]["State"] == st].sort_values("year_month")
    ax.plot(d["year_month"], d["accidents"], color=INK, lw=2, marker="o", ms=4, label="Actual")
    ax.plot(d["year_month"], d["prediction"], color=AQUA, lw=2, marker="o", ms=4, label="Random forest forecast")
    ax.plot(d["year_month"], d["lag1"], color=GREY, lw=1.2, ls="--", label="Naive (last month)")
    ax.set_title(st, fontsize=11); ax.tick_params(axis="x", rotation=45)
h, l = axes.flat[0].get_legend_handles_labels()
fig.suptitle("Monthly accidents: forecast vs actual, busiest 4 states (test period)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.legend(h, l, loc="lower center", ncol=3, frameon=False, fontsize=9)
f_ts = f"{FIG_DIR}/B2_forecast_vs_actual.png"
fig.savefig(f_ts, bbox_inches="tight", facecolor="white"); display(fig)

In [ ]:
# Chart B3: residuals (forecast errors) of the random forest
res = rpred["Random forest"]["accidents"] - rpred["Random forest"]["prediction"]
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.hist(res, bins=40, color=AQUA, edgecolor="white")
ax.axvline(0, color=GREY, ls="--", lw=1.2)
ax.set_title("Random forest forecast errors (actual minus predicted)")
ax.set_xlabel("Error (accidents per state-month)"); ax.set_ylabel("Number of state-months")
f_res = save(fig, "B3_residuals"); display(fig)

In [ ]:
# Record and track Part B
rparams = {"train_until": str(cutoff), "test_until": str(last_month), "inputs": ", ".join(reg_inputs + ["State"])}
for name in ["Linear regression", "Random forest"]:
    record(name, "volume_regression", "test", reg_results[name])
record("Naive: next month = this month", "volume_regression", "test", reg_results["Naive: next month = this month"])
log_run("B_linear_regression", {**rparams, "model": "LinearRegression", "regParam": 0.1, "target": "change from last month"}, reg_results["Linear regression"], [f_avp])
log_run("B_random_forest", {**rparams, "model": "RandomForestRegressor", "numTrees": 100, "maxDepth": 6, "target": "change from last month"},
        reg_results["Random forest"], [f_avp, f_ts, f_res])

# Part C: Save the results and show the project's track record
All model metrics go into `gold_model_metrics`, so the dashboard can show them next to the EDA. We then list every tracked run from MLflow and every quality check in `dq_log` since Checkpoint 3: together they are the audit trail of the project.

In [ ]:
from datetime import datetime
mdf = spark.createDataFrame([(datetime.now(),) + r for r in all_metrics],
    "run_ts timestamp, model string, task string, split string, metric string, value double")
mdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("gold_model_metrics"))
display(spark.table(tbl("gold_model_metrics")).orderBy("task", "model", "metric"))

In [ ]:
if MLFLOW:
    try:
        runs = mlflow.search_runs(order_by=["start_time DESC"])
        keep = [c for c in runs.columns if c in ("tags.mlflow.runName", "start_time") or c.startswith("metrics.")]
        display(spark.createDataFrame(runs[keep].head(20).astype(str)))
    except Exception as e:
        print("Could not list MLflow runs:", str(e)[:150])
display(spark.table(tbl("dq_log")).orderBy("run_ts", "layer", "check_name"))

In [ ]:
# Plain-English summary for the presentation and final report
A_rf, A_lr = results["Random forest"], results["Logistic regression"]
B_rf, B_lr, B_nv = reg_results["Random forest"], reg_results["Linear regression"], reg_results["Naive: next month = this month"]
best_A = "Random forest" if A_rf["roc_auc"] >= A_lr["roc_auc"] else "Logistic regression"
best_B = "Random forest" if B_rf["r2"] >= B_lr["r2"] else "Linear regression"
print("MODEL A - severity risk (test set)")
print(f"  Best model: {best_A}. ROC-AUC {results[best_A]['roc_auc']:.3f} (random guess 0.500); recall {results[best_A]['recall']:.1%}; precision {results[best_A]['precision']:.1%}; F1 {results[best_A]['f1']:.3f}.")
print(f"  Overfitting check: train ROC-AUC {train_auc[best_A]:.3f} vs test {results[best_A]['roc_auc']:.3f}.")
print(f"  Top 3 drivers (random forest): {', '.join(fi['feature'].tolist()[::-1][:3])}.")
print("MODEL B - monthly accident volume by state (last 12 months held out)")
print(f"  Best model: {best_B}. R² {reg_results[best_B]['r2']:.3f}, MAE {reg_results[best_B]['mae']:.1f} accidents, MAPE {reg_results[best_B]['mape_pct']:.1f}%.")
print(f"  Naive forecast: R² {B_nv['r2']:.3f}, MAE {B_nv['mae']:.1f}. Beats naive on MAE: {reg_results[best_B]['mae'] < B_nv['mae']}.")